# APEC rest-frame surrogate pilot

This notebook builds a temperature surrogate from a local AtomDB `_line.fits` and `_coco.fits` pair and compares it with native Plasma.jl. The same rest-frame basis serves Apec, Vapec, Vvapec and their broadened variants on either energy grid. Training, a held-out temperature check, and a fitting benchmark are separate cells.

Use a Julia environment with this Plasma.jl checkout, SpectralFitting 0.7.3 or 0.7.4, NMF, and DataInterpolations. JLD2 is needed only to save a trained basis. Supply an existing SpectralGrouping `joint` before running the objective cells to use its responses and likelihood. The default uses a synthetic spectrum generated by native Bvapec.

The preset covers all continuum and line energies present in a bounded temperature interval. Start with that interval to size the dense training matrices, then increase the range, rank, or continuum resolution in the setup cell.

In [ ]:
using Plasma, LinearAlgebra, Statistics, SHA, Test
import SpectralFitting as SF
import NMF
using DataInterpolations: PCHIPInterpolation

atomdb_prefix = get(ENV, "ATOMDB_PREFIX", joinpath(homedir(), "APED", "apec_v3.1.3"))
bench_db = (@isdefined db) ? db : Plasma.load(AtomDB, atomdb_prefix)

temperature_range = (0.35, 2.5)  # keV
max_rank = 8                    # per element and per contribution
continuum_points = 1536
nmf_iterations = 60
holdout_stride = 4
basis_file = nothing            # optional path for a JLD2 file

coarse_edges = collect(range(0.3, 8.0; length=1201))
grating_edges = reverse(bench_db.constants.kev_angstrom ./ collect(range(1.7, 10.5; length=4401)))
node_ids = findall(t -> temperature_range[1] <= t <= temperature_range[2], bench_db.temperatures)
length(node_ids) >= 5 || error("Choose a temperature range containing at least five AtomDB nodes.")
holdout_stride >= 2 || error("holdout_stride must be at least two.")
max_rank >= 1 || error("max_rank must be positive.")
if !isnothing(basis_file)
    @eval import JLD2
end
display((temperatures_keV=extrema(bench_db.temperatures[node_ids]), nodes=length(node_ids)))

## Train an elemental rest-frame basis

Stack spectra at the selected AtomDB temperatures as columns. Fit a nonnegative factorization for the continuum (including pseudo-continuum) and for resolved line strengths of each element. The continuum uses logarithmic knots plus the tabulated row endpoints, with points just outside each support boundary. Resolved lines retain their original energies. Entries with identical energy and element are summed because they have identical profiles in this model.

The continuum sampling introduces its own approximation. Increase `continuum_points` to check its effect. Setting the rank at least as high as the number of training temperatures uses the original sample matrices as a control.

In [ ]:
function fit_factors(samples, rank, maxiter)
    scale = maximum(samples; init=0.0)
    if iszero(scale)
        return (basis=zeros(size(samples, 1), 0), weights=zeros(0, size(samples, 2)), converged=true)
    end
    if rank >= size(samples, 2)
        n = size(samples, 2)
        return (basis=samples, weights=Matrix{Float64}(I, n, n), converged=true)
    end
    r = min(rank, size(samples, 1))
    fit = NMF.nnmf(samples ./ scale, r; init=:nndsvda, alg=:multmse, maxiter)
    return (basis=fit.W .* scale, weights=fit.H, converged=fit.converged)
end

function element_samples(slices, element, continuum_points)
    continua = [[row for (z, row) in slice.continua if z == element] for slice in slices]
    knots = Float64[]
    for rows in continua, row in rows
        lo, hi = Float64(first(row.energies)), Float64(last(row.energies))
        append!(knots, (prevfloat(lo), lo, hi, nextfloat(hi)))
    end
    if !isempty(knots)
        lo, hi = extrema(knots)
        append!(knots, exp.(range(log(lo), log(hi); length=continuum_points)))
        sort!(unique!(knots))
    end
    continuum = zeros(length(knots), length(slices))
    widths = ones(length(knots))
    for (i, rows) in enumerate(continua), row in rows
        Plasma.add_continuum!(view(continuum, :, i), knots, widths, row, 1.0)
    end

    positions = [findall(==(element), slice.lines.elements) for slice in slices]
    energies = Float64[]
    for (slice, rows) in zip(slices, positions)
        append!(energies, view(slice.lines.energies, rows))
    end
    sort!(unique!(energies))
    index_at_energy = Dict(energy => i for (i, energy) in enumerate(energies))
    lines = zeros(length(energies), length(slices))
    for (i, (slice, rows)) in enumerate(zip(slices, positions)), j in rows
        lines[index_at_energy[slice.lines.energies[j]], i] += slice.lines.emissivities[j]
    end
    return knots, continuum, energies, lines
end

function train_apec(db, ids; rank, continuum_points, maxiter)
    slices = [Plasma.temperature_slice(db, i) for i in ids]
    elements = map(1:30) do z
        knots, continuum, energies, lines = element_samples(slices, z, continuum_points)
        (knots=knots, energies=energies, atomic_numbers=fill(UInt8(z), length(energies)),
         continuum=fit_factors(continuum, rank, maxiter), lines=fit_factors(lines, rank, maxiter))
    end
    return (temperatures=db.temperatures[ids], elements=elements)
end

train(db, ids) = train_apec(db, ids; rank=max_rank, continuum_points, maxiter=nmf_iterations)

function basis_provenance(db, ids)
    checksums = map(path -> bytes2hex(open(sha256, path)), (db.linefile, db.cocofile))
    constants = db.constants
    return (schema=1, checksums=checksums, temperatures=db.temperatures[ids],
            wavelength_scale=constants.kev_angstrom, storage_type=string(typeof(db)), rank=max_rank,
            continuum_points=continuum_points, nmf_iterations=nmf_iterations)
end

function prepare_basis(db, ids, path)
    isnothing(path) && return train(db, ids)
    provenance = basis_provenance(db, ids)
    if isfile(path)
        saved = JLD2.load(path)
        saved["provenance"] == provenance ||
            error("Saved basis settings differ. Choose a new basis_file.")
        return saved["basis"]
    end
    basis = train(db, ids)
    JLD2.jldsave(path; basis, provenance)
    return basis
end

training = @timed prepare_basis(bench_db, node_ids, basis_file)
basis = training.value
display((setup_seconds=training.time, basis_MiB=Base.summarysize(basis)/2.0^20,
         blocks_reaching_iteration_cap=count(f -> !f.converged,
             Iterators.flatten((b.continuum, b.lines) for b in basis.elements))))

## Project the basis onto an observed grid

The prototype reconstructs the two relevant temperature nodes and uses Plasma's midpoint continuum and line-profile routines. Thermal broadening uses the element mass and temperature of each node. The node spectra are then mixed with native linear temperature weights.

The timing below includes reconstruction and projection on each surrogate trial. A later backend can reuse projections for fixed geometry.

In [ ]:
const ApecExtension = Base.get_extension(Plasma, :PlasmaSpectralFittingExt)

function add_element!(output, edges, centers, widths, block, continuum_coeffs, line_coeffs,
        scale, line_widths, abundance_scales)
    if !isempty(continuum_coeffs)
        continuum = (energies=block.knots, density=block.continuum.basis * continuum_coeffs)
        Plasma.add_continuum!(output, centers, widths, continuum, scale)
    end
    if !isempty(line_coeffs)
        lines = (energies=block.energies, emissivities=block.lines.basis * line_coeffs,
                 elements=block.atomic_numbers)
        if isnothing(line_widths)
            Plasma.add_lines!(output, edges, lines, abundance_scales, nothing)
        else
            Plasma.add_broadened_lines!(output, edges, lines, abundance_scales, line_widths, nothing)
        end
    end
    return nothing
end

function add_node!(out, geometry, basis, index, scales, weight, thermal, velocity, constants)
    edges, centers, widths = geometry
    line_widths = thermal || !iszero(velocity) ?
        Plasma.line_width_fractions(basis.temperatures[index], thermal, velocity, constants) : nothing
    weighted = map(scale -> scale * weight, scales)
    for (z, block) in enumerate(basis.elements)
        add_element!(out, edges, centers, widths, block,
            view(block.continuum.weights, :, index), view(block.lines.weights, :, index),
            weighted[z], line_widths, weighted)
    end
end

function surrogate_flux!(out, edges, basis, constants, scales; kT, thermal, velocity, redshift)
    first(basis.temperatures) <= kT <= last(basis.temperatures) ||
        throw(DomainError(kT, "Temperature is outside this notebook's training interval."))
    geometry = Plasma.source_grid(edges, redshift)
    lo, hi, fraction = Plasma.temperature_bracket(basis.temperatures, kT)
    fill!(out, zero(eltype(out)))
    add_node!(out, geometry, basis, lo, scales, 1 - fraction, thermal, velocity, constants)
    if hi != lo
        add_node!(out, geometry, basis, hi, scales, fraction, thermal, velocity, constants)
    end
    out .*= 1e14 / (1 + redshift)
    return out
end

function surrogate_prediction(edges, basis, model)
    numeric = SF.remake_with_parameters(model, SF.get_value.(SF.parameter_vector(model)))
    scales = Plasma.abundance_scales(ApecExtension.model_abundances(numeric), numeric.options)
    flux = zeros(length(edges) - 1)
    surrogate_flux!(flux, edges, basis, numeric.db.constants, scales;
        kT=numeric.kT, thermal=numeric.options.thermal, redshift=numeric.redshift,
        velocity=ApecExtension.model_velocity(numeric))
    return flux .* numeric.K
end

function fractional_l1(predicted, reference)
    denominator = sum(abs, reference)
    return iszero(denominator) ? missing : sum(abs, predicted .- reference) / denominator
end

function band_error(edges, prediction, reference, lo, hi, z)
    centers = (edges[1:end-1] .+ edges[2:end]) .* ((1 + z) / 2)
    keep = (lo .<= centers) .& (centers .<= hi)
    expected = sum(reference[keep])
    return iszero(expected) ? missing : (sum(prediction[keep]) - expected) / expected
end

## Check six APEC variants

Use tabulated and between-node temperatures, redshift and blueshift, tied and individual abundances, and thermal-only, velocity-only, and combined line profiles. Cache-to-direct agreement checks the benchmark setup. Surrogate errors are reported by case and in O VII, O VIII, and Fe-L bands so rank-dependent discrepancies remain visible.

In [ ]:
cases = (
    (Apec,    :Apec,    (; abundance=0.5)),
    (Vapec,   :Vapec,   (; Fe=0.5, O=0.8, trace_abundance=:Fe)),
    (Vvapec,  :Vvapec,  (; Fe=0.5, O=0.8, H=1.2, K_abundance=0.7)),
    (Bapec,   :Bapec,   (; abundance=0.5, velocity=0.0)),
    (Bvapec,  :Bvapec,  (; Fe=0.5, velocity=180.0, thermal_broadening=false)),
    (Bvvapec, :Bvvapec, (; Fe=0.5, O=0.8, K_abundance=0.7, velocity=180.0)),
)
i = length(basis.temperatures) ÷ 2
temperatures_to_check = (basis.temperatures[i], (basis.temperatures[i] + basis.temperatures[i+1]) / 2)
spectrum_checks = NamedTuple[]
@testset "APEC basis projection and cache reference" begin
    for (constructor, name, options) in cases, (grid, edges) in ((:coarse, coarse_edges), (:grating, grating_edges)),
            kT in temperatures_to_check, z in (0.0, -0.003)
        settings = (; kT, redshift=z, K=1e-3, abundance_table=:wilm, options...)
        cached = constructor(bench_db; settings...)
        direct = constructor(bench_db; settings..., cache_spectra=false)
        reference = vec(SF.invokemodel(edges, direct))
        SF.invokemodel(edges, cached)
        @test isapprox(vec(SF.invokemodel(edges, cached)), reference; rtol=1e-10)
        prediction = surrogate_prediction(edges, basis, cached)
        @test all(x -> isfinite(x) && x >= 0, prediction)
        push!(spectrum_checks, (; model=name, grid, kT, z, relative_l1=fractional_l1(prediction, reference),
            O_VII=band_error(edges, prediction, reference, 0.55, 0.58, z),
            O_VIII=band_error(edges, prediction, reference, 0.64, 0.67, z),
            Fe_L=band_error(edges, prediction, reference, 0.72, 1.20, z)))
    end
end
display(spectrum_checks)

## Compare interpolation at held-out AtomDB temperatures

Exclude interior temperatures from factorization and fit each coefficient with a shape-preserving cubic in log temperature. Evaluate unbroadened spectra here to isolate the temperature approximation. Compare the cubic and linear surrogate with exact node spectra and with native linear interpolation using the same retained nodes.

In [ ]:
function prepare_curves(basis)
    logT = log.(basis.temperatures)
    return map(basis.elements) do block
        (continuum=[PCHIPInterpolation(block.continuum.weights[j, :], logT) for j in axes(block.continuum.weights, 1)],
         lines=[PCHIPInterpolation(block.lines.weights[j, :], logT) for j in axes(block.lines.weights, 1)])
    end
end

function cubic_prediction(edges, basis, curves, kT, scales)
    source, centers, widths = Plasma.source_grid(edges, 0.0)
    out = zeros(length(widths))
    logT = log(kT)
    for (z, (block, curve)) in enumerate(zip(basis.elements, curves))
        add_element!(out, source, centers, widths, block, [f(logT) for f in curve.continuum],
                     [f(logT) for f in curve.lines], scales[z], nothing, scales)
    end
    return out .* 1e14
end

held_ids = node_ids[2:holdout_stride:end-1]
retained_ids = setdiff(node_ids, held_ids)
held_basis = train(bench_db, retained_ids)
curves = prepare_curves(held_basis)
scales = ntuple(_ -> 1.0, 30)
heldout_results = map(held_ids) do index
    kT = bench_db.temperatures[index]
    edges = grating_edges
    reference = Plasma.apec(edges, bench_db; kT)
    lo, hi, weight = Plasma.temperature_bracket(held_basis.temperatures, kT)
    retained_linear = (1 - weight) .* Plasma.apec(edges, bench_db; kT=held_basis.temperatures[lo]) .+
        weight .* Plasma.apec(edges, bench_db; kT=held_basis.temperatures[hi])
    surrogate_linear = zeros(length(edges) - 1)
    surrogate_flux!(surrogate_linear, edges, held_basis, bench_db.constants, scales;
        kT, thermal=false, velocity=0.0, redshift=0.0)
    cubic = cubic_prediction(edges, held_basis, curves, kT, scales)
    (; kT, native_linear=fractional_l1(retained_linear, reference),
     surrogate_linear=fractional_l1(surrogate_linear, reference), surrogate_cubic=fractional_l1(cubic, reference))
end
display(heldout_results)

## Compare fitted objectives and gradients

This small SpectralFitting adapter gives the surrogate Bvapec the same free parameters as the native reference. If `joint` is available, use its observed data and keep its response folding and statistic. Otherwise, use a noiseless native Bvapec target and a chi-squared objective with supplied errors. Compare timings both with fixed geometry and with redshift and velocity free. The native cache can fall back to direct synthesis when these parameters vary or carry active derivatives.

In [ ]:
struct NotebookBvapec{T,B,O,C} <: SF.AbstractSpectralModel{T,SF.Additive}
    basis::B
    options::O
    constants::C
    K::T
    kT::T
    Fe::T
    redshift::T
    velocity::T
end

function Base.copy(model::NotebookBvapec{<:SF.FitParam})
    return NotebookBvapec(model.basis, model.options, model.constants,
        copy(model.K), copy(model.kT), copy(model.Fe), copy(model.redshift), copy(model.velocity))
end

function SF.invoke!(output, edges, model::NotebookBvapec)
    abundances = Plasma.partial_abundances(
        map(name -> name === :Fe ? model.Fe : one(model.Fe), Plasma.VAPEC_NAMES), model.options.trace)
    scales = Plasma.abundance_scales(abundances, model.options)
    return surrogate_flux!(output, edges, model.basis, model.constants, scales;
        kT=model.kT, velocity=model.velocity, redshift=model.redshift, thermal=model.options.thermal)
end

nodes = basis.temperatures
ilo, ihi = 2, length(nodes)-1
temperature_lo = nodes[ilo]
temperature_hi = nodes[ihi]
start_temperature = (nodes[i] + nodes[i+1]) / 2
start = (;
    K=SF.FitParam(1e-3; lower_limit=2e-4, upper_limit=3e-3),
    kT=SF.FitParam(start_temperature; lower_limit=temperature_lo, upper_limit=temperature_hi),
    Fe=SF.FitParam(0.7; lower_limit=0.1, upper_limit=2.0),
    redshift=SF.FitParam(0.0012; lower_limit=-0.006, upper_limit=0.006),
    velocity=SF.FitParam(170.0; lower_limit=50.0, upper_limit=500.0),
)
cached_model = Bvapec(bench_db; start...)
direct_model = Bvapec(bench_db; start..., cache_spectra=false)
surrogate_model = NotebookBvapec(basis, cached_model.options, bench_db.constants,
    start.K, start.kT, start.Fe, start.redshift, start.velocity)

use_joint = (@isdefined joint)
if use_joint
    @eval using SpectralGrouping: grid_statistic
    fit_data, statistic = joint, grid_statistic(joint)
else
    target = Bvapec(bench_db; K=1.1e-3, kT=0.85 * start_temperature + 0.15 * temperature_hi,
        Fe=0.83, redshift=-0.002, velocity=220.0, cache_spectra=false)
    truth = vec(SF.invokemodel(grating_edges, target))
    variance = fill((maximum(truth) / 80)^2, length(truth))
    fit_data = SF.BinnedData(grating_edges, truth; codomain_variance=variance)
    statistic = SF.ChiSquared()
end
fit_config(model) = SF.FittingConfig(SF.FittingProblem(model => fit_data); stat=statistic)
configs = map(fit_config, (cached_model, direct_model, surrogate_model))
labels = (:cache, :direct, :surrogate)
initial = SF.get_value.(configs[1].u0)
@test all(c -> SF.get_value.(c.u0) == initial, configs)
objectives = map(c -> x -> SF.measure_objective!(c, x, 1), configs)
FD = SF.ForwardDiff
trial_points = [initial, initial .* [1.02, 1.01, 0.95, 0.9, 1.1], initial .* [0.97, 0.99, 1.05, 1.1, 0.95]]

function centered_gradient(f, point)
    return map(eachindex(point)) do j
        step = 1e-6 * max(abs(point[j]), 1e-3)
        plus, minus = copy(point), copy(point)
        plus[j] += step
        minus[j] -= step
        (f(plus) - f(minus)) / (2 * step)
    end
end

gradient_checks = map(trial_points) do point
    for objective in objectives
        @test isfinite(objective(point))
        @test all(isfinite, FD.gradient(objective, point))
    end
    @test isapprox(objectives[1](point), objectives[2](point); rtol=1e-10, atol=1e-10)
    @test isapprox(FD.gradient(objectives[1], point), FD.gradient(objectives[2], point); rtol=1e-8, atol=1e-8)
    exact = FD.gradient(objectives[2], point)
    surrogate = FD.gradient(objectives[3], point)
    finite_difference = centered_gradient(objectives[3], point)
    counts = map(c -> copy(only(SF.calculate_objective!(c, point))), configs)
    (; point, statistic_delta=objectives[3](point)-objectives[2](point),
     folded_l1=fractional_l1(counts[3], counts[2]), surrogate_exact_gradient_l1=fractional_l1(surrogate, exact),
     autodiff_finite_difference_l1=fractional_l1(surrogate, finite_difference))
end
display(gradient_checks)

In [ ]:
function milliseconds_per_trial(f, points; repeats=5)
    foreach(f, points)
    samples = [(@elapsed foreach(f, points)) for _ in 1:repeats]
    return 1000 * median(samples) / length(points)
end

function source_evaluator(model, edges)
    parameters = SF.parameter_vector(model)
    values = SF.get_value.(parameters)
    free = findall(p -> !p.frozen, parameters)
    output = zeros(length(edges) - 1, 1)
    return function evaluate(x)
        values[free] .= x
        return SF.invokemodel!(output, edges, model, values)
    end
end

fit_edges = use_joint ? joint.model_edges : grating_edges
sources = map(model -> source_evaluator(model, fit_edges), (cached_model, direct_model, surrogate_model))
fixed_points = [point[1:3] for point in trial_points]
timings = NamedTuple[]
for (backend, objective, source) in zip(labels, objectives, sources)
    fixed = x -> objective(vcat(x, initial[4:5]))
    fixed_source = x -> source(vcat(x, initial[4:5]))
    for (geometry, f, source_trial, points) in
            ((:fixed, fixed, fixed_source, fixed_points), (:free, objective, source, trial_points))
        push!(timings, (; backend, geometry, source_ms=milliseconds_per_trial(source_trial, points),
            objective_ms=milliseconds_per_trial(f, points),
            gradient_ms=milliseconds_per_trial(x -> FD.gradient(f, x), points)))
    end
end
display(timings)

## Optional bounded refits

Enable this cell with Optimization and OptimizationOptimJL installed. These fits use identical starting points and bounds. With synthetic data, compare estimates with the known generating parameters. With joint data, compare both the fitted parameters and the native objective evaluated at each solution.

In [ ]:
run_fits = false

if run_fits
    @eval import Optimization as Opt
    @eval using OptimizationOptimJL: Optim
    scales = max.(abs.(initial), 1e-3)
    lower = SF.get_lowerlimit.(configs[1].u0)
    upper = SF.get_upperlimit.(configs[1].u0)
    for (backend, objective) in zip(labels, objectives)
        f = Opt.OptimizationFunction((x, _) -> objective(x .* scales), Opt.AutoForwardDiff())
        problem = Opt.OptimizationProblem(f, initial ./ scales; lb=lower ./ scales, ub=upper ./ scales)
        Opt.solve(problem, Optim.Fminbox(Optim.LBFGS()))
        result = @timed Opt.solve(problem, Optim.Fminbox(Optim.LBFGS()))
        values = result.value.u .* scales
        display((; backend, status=result.value.retcode, seconds=result.time, parameters=values,
            objective=objective(values), native_objective=objectives[2](values)))
    end
end